#Завдання 1. Setup і typed-схема Olist

In [ ]:
!pip install -q --no-cache-dir pgembed psycopg2-binary psycopg[binary] sqlalchemy pandas kagglehub

import sys
import pgembed
sys.modules['pgserver'] = pgembed

import os
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text
import kagglehub

pg = pgserver.get_server('/tmp/hw4_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print("PostgreSQL Version:", conn.execute(text('SELECT version()')).scalar_one())

In [ ]:
%load_ext sql
%sql {engine.url}

In [ ]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [ ]:
dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print("Dataset directory:", dataset_dir)

files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
    'olist_order_reviews_raw': 'olist_order_reviews_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    print(f"Loaded {table_name}: {df.shape}")

In [ ]:
%%sql
DROP TABLE IF EXISTS customer_segments    CASCADE;
DROP TABLE IF EXISTS seller_score         CASCADE;
DROP TABLE IF EXISTS seller_alerts        CASCADE;
DROP TABLE IF EXISTS olist_customer_dim   CASCADE;
DROP TABLE IF EXISTS olist_order_reviews  CASCADE;
DROP TABLE IF EXISTS olist_order_items    CASCADE;
DROP TABLE IF EXISTS olist_orders         CASCADE;
DROP TABLE IF EXISTS olist_customers      CASCADE;
DROP TABLE IF EXISTS olist_products       CASCADE;
DROP TABLE IF EXISTS olist_sellers        CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE INDEX idx_olist_customers_unique_id
    ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE INDEX idx_olist_order_items_seller_order
    ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id            BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id                TEXT,
    order_id                 TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score             SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title     TEXT,
    review_comment_message   TEXT,
    review_creation_date     TIMESTAMP,
    review_answer_timestamp  TIMESTAMP
);

CREATE INDEX idx_olist_order_reviews_order_id
    ON olist_order_reviews (order_id);


In [ ]:
%%sql
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
SELECT
    review_id,
    order_id,
    review_score::SMALLINT,
    review_comment_title,
    review_comment_message,
    review_creation_date::TIMESTAMP,
    review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;


#Завдання 2. DML-запити з PostgreSQL-фічами

2.1. INSERT ... RETURNING: створення alert для продавця з найбільшою кількістю запізнень

In [ ]:
%%sql
DROP TABLE IF EXISTS seller_alerts CASCADE;

CREATE TABLE seller_alerts (
    alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type TEXT NOT NULL CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
    severity   SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details    TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
late_sellers AS (
    SELECT
        so.seller_id,
        COUNT(DISTINCT o.order_id) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o
        ON o.order_id = so.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
    GROUP BY so.seller_id
    ORDER BY late_orders DESC, so.seller_id
    LIMIT 1
)
INSERT INTO seller_alerts (seller_id, alert_type, severity, details)
SELECT
    seller_id,
    'late_delivery',
    4,
    'Seller has ' || late_orders || ' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING alert_id, seller_id, alert_type, severity, created_at;

Створює таблицю аудиту seller_alerts та записує туди інформацію про продавця з найбільшою кількістю затримок доставки (late_delivery).
Завдяки конструкції RETURNING ми одразу бачимо згенеровані дані нового рядка прямо у виводі ячейки, що слугує готовим audit-логом виконаної операції без необхідності писати окремий SELECT-запит.

2.2. INSERT ... ON CONFLICT DO UPDATE: idempotent upsert агрегованих seller-score

In [ ]:
%%sql
DROP TABLE IF EXISTS seller_score CASCADE;

CREATE TABLE seller_score (
    seller_id   TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
    avg_rating  NUMERIC(3, 2),
    n_reviews   INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
    tier        TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
    updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT
        so.seller_id,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
        COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
)
INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
SELECT
    seller_id,
    avg_rating,
    n_reviews,
    CASE
        WHEN avg_rating IS NULL THEN NULL
        WHEN avg_rating >= 4.5 THEN 'gold'
        WHEN avg_rating >= 3.5 THEN 'silver'
        ELSE 'bronze'
    END AS tier
FROM seller_rating
ON CONFLICT (seller_id) DO UPDATE SET
    avg_rating = EXCLUDED.avg_rating,
    n_reviews  = EXCLUDED.n_reviews,
    tier       = EXCLUDED.tier,
    updated_at = NOW()
RETURNING seller_id, avg_rating, n_reviews, tier, updated_at;

Чому seller_orders використовує SELECT DISTINCT: Оскільки одне замовлення (order_id) може містити кілька позицій від одного продавця (order_item_id), звичайний JOIN здублював би замовлення. DISTINCT гарантує, що кожне замовлення продавця враховується рівно один раз.

Чому review потрібно спочатку агрегувати на рівні order_id: У датасеті Olist відгуки прив'язані до замовлення загалом, а не до конкретної позиції чи продавця. Спочатку вираховується середній бал відгуку для кожного order_id, щоб уникнути перекосів.

Чому ON CONFLICT (seller_id) DO UPDATE робить запуск повторюваним: Це ключова фіча ідемпотентності. Якщо запит запускається повторно, PostgreSQL не викидає помилку дублювання первинного ключа, а оновлює (UPDATE) існуючі рядки новими значеннями.

Чому sellers без review мають NULL / 0: Продавці, у яких не було замовлень із відгуками, потрапляють у вибірку завдяки LEFT JOIN, але отримують нульові або порожні агрегати (NULL для рейтингу, 0 для кількості відгуків).

Обмеження датасету: В Olist відгук ставиться на замовлення, а не на окремого продавця. Якщо в замовленні товари від кількох продавців, цей спільний рейтинг технічно присвоюється кожному з них — це прийнятне навчальне спрощення.

2.3. UPDATE ... FROM: оновлення прапорця запізнення в olist_orders

In [ ]:
%%sql
ALTER TABLE olist_orders
    ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;

WITH delivery_flags AS (
    SELECT
        order_id,
        (
            order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE
        ) AS is_late_calc
    FROM olist_orders
)
UPDATE olist_orders AS o
SET is_late = f.is_late_calc
FROM delivery_flags AS f
WHERE f.order_id = o.order_id
RETURNING o.order_id, o.customer_id, o.is_late;

Додає до таблиці замовлень булевий стовпчик is_late та масово оновлює його значення на основі реальної та очікуваної дат доставки.
Запит повністю ідемпотентний. Якщо виконати його повторно, значення is_late буде перераховано за актуальними датами з урахуванням логіки виразу, а не просто сліпо перезаписано статечими булевими знаками.

2.4. DELETE ... RETURNING: контрольоване видалення застарілих alert-ів

In [ ]:
%%sql
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at;

Видаляє з таблиці seller_alerts записи, які були створені понад 30 днів тому.
Чому порожній результат є нормальним, якщо в таблиці немає алертів старших за 30 днів (наприклад, вони щойно створені в рамках поточного запуску ноутбука), DELETE не видалить жодного рядка і поверне пусту таблицю. Це стандартна поведінка очищення даних, яка не свідчить про помилку.

#Завдання 3. CREATE TABLE customer_segments

In [ ]:
%%sql
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT REFERENCES olist_customers(customer_id),
    customer_state     CHAR(2),
    customer_city      TEXT
);

INSERT INTO olist_customer_dim (
    customer_unique_id,
    latest_customer_id,
    customer_state,
    customer_city
)
SELECT DISTINCT ON (c.customer_unique_id)
    c.customer_unique_id,
    c.customer_id AS latest_customer_id,
    c.customer_state,
    c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
ORDER BY
    c.customer_unique_id,
    o.order_purchase_timestamp DESC NULLS LAST,
    c.customer_id;

In [ ]:
%%sql
CREATE TABLE customer_segments (
    segment_id          BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    customer_unique_id  TEXT NOT NULL,
    segment_name        TEXT NOT NULL
        CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),

    rfm_score           SMALLINT NOT NULL
        CHECK (rfm_score BETWEEN 1 AND 5),

    recency_days        INTEGER
        CHECK (recency_days IS NULL OR recency_days >= 0),

    monetary_value      NUMERIC(12, 2) NOT NULL DEFAULT 0
        CHECK (monetary_value >= 0),

    n_orders            INTEGER NOT NULL DEFAULT 0
        CHECK (n_orders >= 0),

    assigned_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),

    UNIQUE (customer_unique_id)
);

ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id)
    REFERENCES olist_customer_dim(customer_unique_id)
    ON DELETE CASCADE;

WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp
),
customer_stats AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
        MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c
        ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot
        ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
segmented AS (
    SELECT
        cs.customer_unique_id,
        cs.n_orders,
        cs.monetary_value,
        CASE
            WHEN cs.last_order_date IS NULL THEN NULL
            ELSE (rd.as_of_date - cs.last_order_date)::INTEGER
        END AS recency_days,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 'VIP'
            WHEN cs.last_order_date IS NOT NULL
                 AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                 AND cs.n_orders = 1 THEN 'new'
            WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
            ELSE 'inactive'
        END AS segment_name,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 5
            WHEN cs.monetary_value >= 500  THEN 4
            WHEN cs.n_orders >= 2          THEN 3
            WHEN cs.n_orders = 1           THEN 2
            ELSE 1
        END::SMALLINT AS rfm_score
    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
)
INSERT INTO customer_segments (
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
)
SELECT
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
FROM segmented
ON CONFLICT (customer_unique_id) DO UPDATE SET
    segment_name   = EXCLUDED.segment_name,
    rfm_score      = EXCLUDED.rfm_score,
    recency_days   = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders       = EXCLUDED.n_orders,
    assigned_at    = NOW()
RETURNING
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders;

In [ ]:
%%sql
SELECT
    conname,
    contype,
    pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname;

1. У базі даних Olist ідентифікатор customer_id створюється під кожне окреме замовлення, тоді як customer_unique_id відповідає реальній людині. Якщо рахувати RFM-метрики за customer_id, історія повторних покупок одного покупця розбиватиметься на різні рядки, а частота (frequency) незмінно дорівнюватиме 1.

2. Оскільки таблиця olist_customers містить рядки на рівні замовлень (customer_id це первинний ключ, а customer_unique_id повторюється), неможливо напряму посилатися на customer_unique_id як на зовнішній ключ, якщо він не є унікальним. Дедуплікований вимір olist_customer_dim робить customer_unique_id унікальним первинним ключем, що дозволяє безпечно зв'язати його через FOREIGN KEY.

3. Оскільки датасет Olist є історичним (дані охоплюють періоди 2016–2018 років), використання поточниї дати (CURRENT_DATE) призвело б до того, що показник давності (recency_days) для всіх клієнтів становив би кілька тисяч днів. Обчислення reference_date як наступного дня після максимальної дати замовлення в базі гарантує коректність і стабільність історичного аналізу незалежно від дати запуску запиту.

4. Конструкція ON CONFLICT (customer_unique_id) DO UPDATE перетворює звичайний INSERT на ідемпотентний upsert. За повторного запуску запиту PostgreSQL не повертає помилку порушення унікальності, а оновлює існуючі записи новими розрахованими показниками сегментації та RFM-балів.

#Завдання 4. JOIN-запити з business-interpretation

Приклад 4.1. INNER JOIN: топ категорій за revenue і seller-state

In [ ]:
%%sql
SELECT
    p.product_category_name,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON o.order_id = oi.order_id
JOIN olist_products AS p
    ON p.product_id = oi.product_id
JOIN olist_sellers AS s
    ON s.seller_id = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name, s.seller_state
ORDER BY revenue DESC NULLS LAST
LIMIT 5;

Які категорії товарів і штати продавців дають найбільший прибуток (revenue) у доставлених замовленнях?

- Запит використовує INNER JOIN між чотирма ключовими таблицями бази даних та фільтрує лише успішно доставлені замовлення (delivered). Результати демонструють, що лідерами за загальним обсягом виручки є певні товарні категорії, які відправляються переважно з економічно розвиненого штату Сан-Паулу (SP). Це ключова інформація для оптимізації логістичних ланцюгів та розміщення регіональних складів.

Приклад 4.2. LEFT JOIN + COALESCE: customer-level features без множення review

In [ ]:
%%sql
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_unique_id
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
customer_stats AS (
    SELECT
        ot.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review_score
    FROM order_totals AS ot
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = ot.order_id
    GROUP BY ot.customer_unique_id
)
SELECT
    cd.customer_unique_id,
    cd.customer_state,
    COALESCE(cs.n_orders, 0) AS n_orders,
    COALESCE(cs.total_spend, 0)::NUMERIC(12, 2) AS total_spend,
    COALESCE(cs.avg_review_score, 0)::NUMERIC(3, 2) AS avg_review_score
FROM olist_customer_dim AS cd
LEFT JOIN customer_stats AS cs
    ON cs.customer_unique_id = cd.customer_unique_id
ORDER BY total_spend DESC NULLS LAST
LIMIT 5;

Які customer-level features можна побудувати для реального покупця на рівні customer_unique_id?

- Використання попередньої агрегації в CTE review_per_order захищає від критичної помилки: множення середнього бала відгуку через позиції замовлення (order_items). Завдяки LEFT JOIN з дедуплікованим виміром клієнтів (olist_customer_dim) та функції COALESCE, ми отримуємо повний профіль кожного реального покупця з нульовими показниками замість NULL для тих, хто не здійснював покупок або не залишав оцінок.

Приклад 4.3. FULL OUTER JOIN: які штати присутні серед клієнтів і продавців

In [ ]:
%%sql
WITH customer_states AS (
    SELECT customer_state AS state, COUNT(*) AS n_customers
    FROM olist_customers
    GROUP BY customer_state
), seller_states AS (
    SELECT seller_state AS state, COUNT(*) AS n_sellers
    FROM olist_sellers
    GROUP BY seller_state
)
SELECT
    COALESCE(c.state, s.state) AS state,
    COALESCE(c.n_customers, 0) AS n_customers,
    COALESCE(s.n_sellers, 0) AS n_sellers,
    CASE
        WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
        WHEN c.state IS NOT NULL THEN 'customers_only'
        ELSE 'sellers_only'
    END AS state_presence
FROM customer_states AS c
FULL OUTER JOIN seller_states AS s
    ON s.state = c.state
ORDER BY state
LIMIT 5;

Які штати представлені тільки серед клієнтів, тільки серед продавців або в обох групах?

- Конструкція FULL OUTER JOIN дозволяє зіставити дві незалежні множини (регіони покупців і продавців) без втрати інформації з жодної зі сторін. Колонка state_presence демонструє географічний розподіл платформи: центральні штати охоплені повністю ('both'), проте є регіони, де наявні лише покупці ('customers_only'), що вказує на потенціал для залучення локальних продавців і зменшення витрат на міжрегіональну доставку.

Приклад 4.4. SELF JOIN: пари продавців з одного штату, але з різних міст

In [ ]:
%%sql
SELECT
    s1.seller_id AS seller_a,
    s2.seller_id AS seller_b,
    s1.seller_state,
    s1.seller_city AS city_a,
    s2.seller_city AS city_b
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_state = s2.seller_state
   AND s1.seller_id < s2.seller_id
   AND s1.seller_city IS DISTINCT FROM s2.seller_city
ORDER BY s1.seller_state, seller_a, seller_b
LIMIT 5;

Які seller-пари можуть бути кандидатами для регіонального порівняння в межах одного штату?

- SELF JOIN таблиці продавців із суворими умовами нерівності (s1.seller_id < s2.seller_id) запобігає дублюванню пар і самопорівнянню, а умова IS DISTINCT FROM залишає тільки продавців із різних міст одного штату. Такі пари створюють якісну базу для бенчмаркінгу ефективності логістики, часу виконання замовлень та конкурентного середовища на рівні регіону.

Приклад 4.5. Anti-join: замовлення без відгуку

In [ ]:
%%sql
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp
FROM olist_orders AS o
LEFT JOIN olist_order_reviews AS rv
    ON rv.order_id = o.order_id
WHERE rv.review_row_id IS NULL
ORDER BY o.order_purchase_timestamp
LIMIT 5;

Які замовлення не мають жодного review-запису?

- Interpretation: Класичний anti-join (LEFT JOIN у поєднанні з перевіркою WHERE rv.review_row_id IS NULL) виділяє «сліпу зону» платформи - транзакції, на які покупці не залишили жодного зворотного зв'язку. Аналіз цієї вибірки допомагає виявити проблеми з комунікацією після покупки, скасовані замовлення або категорії товарів, де рівень залученості клієнтів до оцінювання є найнижчим.

#Завдання 5. GROUP BY, HAVING і агрегати

Приклад 5.1. Revenue per product category з фільтром по revenue

In [ ]:
%%sql
SELECT
    p.product_category_name,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS gross_revenue,
    SUM(oi.freight_value)::NUMERIC(12, 2) AS freight,
    AVG(oi.price)::NUMERIC(8, 2) AS avg_item_price
FROM olist_order_items AS oi
JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY p.product_category_name
HAVING SUM(oi.price) > 10000
ORDER BY gross_revenue DESC NULLS LAST
LIMIT 5;

Приклад 5.2. AOV per customer-state тільки для штатів із достатньою кількістю замовлень

In [ ]:
%%sql
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_state,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_state
)
SELECT
    customer_state,
    COUNT(*) AS n_orders,
    SUM(order_total)::NUMERIC(12, 2) AS gmv,
    (SUM(order_total) / NULLIF(COUNT(*), 0))::NUMERIC(8, 2) AS aov
FROM order_totals
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY aov DESC NULLS LAST
LIMIT 5;

Приклад 5.3. Категорії, які купував реальний покупець, через STRING_AGG

In [ ]:
%%sql
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS total_spend,
    STRING_AGG(
        DISTINCT COALESCE(p.product_category_name, '(unknown)'),
        ', ' ORDER BY COALESCE(p.product_category_name, '(unknown)')
    ) AS categories
FROM olist_customers AS c
JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
JOIN olist_order_items AS oi
    ON oi.order_id = o.order_id
LEFT JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 1
ORDER BY total_spend DESC NULLS LAST
LIMIT 5;

WHERE застосовується до агрегації даних. Він фільтрує вихідні рядки у таблиці або результаті з'єднання (JOIN) ще до того, як спрацюють функції на кшталт SUM(), COUNT() чи AVG().

HAVING застосовується після агрегації даних. Він використовується для фільтрації вже сформованих груп (результатів після GROUP BY) на основі обчислених агрегатних значень (наприклад, перевірка умов на зразок HAVING SUM(oi.price) > 10000 або HAVING COUNT(*) >= 100).

#Завдання 6. Set operations, підзапит/window і Reflection

Приклад 6.1. Усі state-коди, що є у клієнтів або продавців

In [ ]:
%%sql
SELECT customer_state AS state, 'customer' AS role
FROM olist_customers

UNION

SELECT seller_state AS state, 'seller' AS role
FROM olist_sellers

ORDER BY state, role;

Обрано UNION. Цей оператор множин дозволяє об'єднати списки штатів клієнтів та продавців в єдину таблицю з автоматичним видаленням дублікатів. Додавання статичних текстових міток ('customer' та 'seller') допомагає чітко розрізнити роль регіону в кожному рядку.

Приклад 6.2. Штати, у яких є і клієнти, і продавці

In [ ]:
%%sql
SELECT customer_state AS state
FROM olist_customers

INTERSECT

SELECT seller_state AS state
FROM olist_sellers

ORDER BY state;

Обрано INTERSECT. Цей оператор виконує перетин множин і виводить лише ті штати, які одночасно існують і в таблиці клієнтів, і в таблиці продавців, дозволяючи знайти повністю перекриті регіони присутності.

Приклад 6.3. Найновіше замовлення кожного реального покупця через ROW_NUMBER()

In [ ]:
%%sql
WITH ranked_orders AS (
    SELECT
        c.customer_unique_id,
        o.customer_id,
        o.order_id,
        o.order_purchase_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC
        ) AS rn
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
)
SELECT
    customer_unique_id,
    customer_id,
    order_id,
    order_purchase_timestamp
FROM ranked_orders
WHERE rn = 1
ORDER BY customer_unique_id
LIMIT 20;

## Рефлексія (Reflection)

1. **Який тип JOIN був найскладнішим і чому?**

   Найскладнішим виявився `LEFT JOIN` у поєднанні з попередньою агрегацією відгуків та позицій замовлень (`order_items`), оскільки потрібно було ретельно контролювати гранулярність даних, щоб уникнути помилкового дублювання рядків і спотворення кінцевих метрик.

2. **Який запит найскладніше було сформулювати як business-question?**

   Запит для побудови агрегованих ознак на рівні унікального користувача (`customer_unique_id`) з розрахунком RFM-показників, оскільки він потребував багаторівневої структури через використання кількох CTE.

3. **Яка PostgreSQL-фіча з Теми 4 здається найбільш корисною для production-ETL?**

   `ON CONFLICT (upsert)` тому що вона забезпечує ідемпотентність повторюваних ETL-запусків, дозволяючи оновлювати існуючі записи без необхідності повного очищення та перезавантаження таблиць.

4. **У яких місцях такого e-commerce pipeline ви б використали window functions у реальному ML-проєкті?**

   Для визначення першої/останньої транзакції користувача, розрахунку (running totals), а також для обчислення інтервалів між послідовними покупками (що критично важливо для моделей прогнозування відтоку клієнтів).

5. **Які data-quality проблеми ви помітили в Olist-таблицях?**

   Наявність пропущених значень у назвах категорій товарів (`product_category_name`), затримки доставки порівняно з прогнозованими датами, а також те, що значна частина замовлень взагалі не містить жодних відгуків від покупців.

6. **Чому для customer-level ML-фіч у цьому ДЗ використано `customer_unique_id`, а не `customer_id`?**

   Тому що `customer_id` в Olist унікальний лише в межах одного конкретного замовлення, тоді як `customer_unique_id` ідентифікує реальну людину, що дозволяє коректно агрегувати історію її повторних покупок (frequency, monetary).

7. **Чому перед обчисленням `seller_score` потрібно було привести дані до грануляності `seller_id × order_id`?**

   Оскільки відгуки в Olist прив'язані до замовлення загалом, а не до окремих товарів, одне замовлення з кількома позиціями від одного продавця призвело б до неправомірного дублювання балів. Попередня агрегація на рівні замовлення захищає від перекосів у рейтингу продавця.